# Version 2 Data Validation

This notebook reviews the automated Checkpoint 37 outputs. The calculations are created by `src/validate_v2_attrition_data.py`; the notebook is a readable inspection layer rather than a second implementation.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "v2_validation"

def read_output(filename):
    return pd.read_csv(OUTPUT_DIR / filename)

## 1. Validation status

Every required integrity, calibration, signal, hierarchy, and leakage check must pass before the project proceeds to temporal dataset construction.

In [2]:
validation = read_output("validation_summary.csv")
validation

,check,status,observed,requirement,details
0,Workforce row count,PASS,10000,"10,000",The Version 2 workforce preserves the designed...
1,Unique employee IDs,PASS,10000,"10,000 unique IDs",Each employee must have exactly one workforce ...
2,Valid termination dates,PASS,2591,Every exit between hire and as-of date,No employee may terminate before hire or after...
3,No post-employment records,PASS,0,0,"{'compensation': 0, 'performance': 0, 'trainin..."
4,Termination event reconciliation,PASS,2591,2591,Each terminated employee has one event; active...
5,Active manager status,PASS,0,0 active employees with inactive managers,Manager exits must trigger valid reassignment.
6,Manager department consistency,PASS,0,0 mismatches,Active employees and current managers remain i...
7,Protected hierarchy levels,PASS,0,0,"Protected levels: ['Department Head', 'Senior ..."
8,Nonzero team-manager attrition,PASS,276,> 0,Version 2 removes the all-leaders-active Versi...
9,Snapshot positive rate,PASS,11.9051%,7%–13%,This is the twelve-month forward attrition rat...


In [3]:
validation["status"].value_counts()

status
PASS    19
Name: count, dtype: int64

## 2. Version 1 versus Version 2

Cumulative attrition and the twelve-month modeling target are different quantities. The comparison keeps them on separate rows.

In [4]:
version_comparison = read_output("v1_v2_comparison.csv")
version_comparison

,metric,version_1,version_2,absolute_change,relative_change
0,Total employees,10000.000000,10000.000000,0.000000,0.000000
1,Cumulative terminations,1713.000000,2591.000000,878.000000,0.512551
2,Cumulative attrition rate,0.171300,0.259100,0.087800,0.512551
3,Voluntary share,0.758319,0.711694,-0.046624,-0.061484
4,Leadership terminations,0.000000,276.000000,276.000000,NaN
5,Snapshot eligible employees,7386.000000,6745.000000,-641.000000,-0.086786
6,Snapshot positive cases,627.000000,803.000000,176.000000,0.280702
7,Snapshot positive rate,0.084890,0.119051,0.034161,0.402417
8,Manager reassignments after exit,0.000000,1913.000000,1913.000000,NaN


In [5]:
snapshot_summary = read_output("snapshot_summary.csv")
snapshot_summary

,metric,value
0,Snapshot date,2025-06-30
1,Prediction end date,2026-06-30
2,Eligible employees,6745
3,Positive cases,803
4,Negative cases,5942
5,Positive rate,0.11905114899925871


## 3. Organizational and department outcomes

Version 2 permits team-manager exits while protecting department heads and senior managers. This removes the Version 1 artifact in which every leader was permanently active.

In [6]:
organizational = read_output("attrition_by_organizational_level.csv")
departments = read_output("attrition_by_department.csv")
display(organizational)
display(departments)

,organizational_level,headcount,terminations,cumulative_attrition_rate
0,Individual Contributor,8796,2315,0.263188
1,Team Manager,1100,276,0.250909
2,Department Head,8,0,0.000000
3,Senior Manager,96,0,0.000000


,department_name,headcount,terminations,cumulative_attrition_rate
0,Customer Support,800,256,0.320000
1,Finance,800,221,0.276250
2,Human Resources,701,190,0.271041
3,Manufacturing,2499,667,0.266907
4,Information Technology,1000,255,0.255000
5,Sales,1000,250,0.250000
6,Supply Chain,1200,284,0.236667
7,Engineering,2000,468,0.234000


## 4. Observable signal checks

No single feature should nearly determine the outcome. At the same time, a simple cross-validated model should detect moderate signal.

In [7]:
numeric_signals = read_output("numeric_signal_checks.csv")
numeric_signals.head(12)

,feature,non_missing_records,no_attrition_mean,attrition_mean,mean_difference,standardized_mean_difference,point_biserial_correlation,absolute_correlation
0,performance_rating,4924,3.564006,3.362373,-0.201633,-0.397724,-0.129175,0.129175
1,average_performance_rating,4924,3.546564,3.364477,-0.182087,-0.371895,-0.120786,0.120786
2,goal_completion,4924,92.807637,89.545424,-3.262214,-0.330203,-0.107245,0.107245
3,no_prior_promotion,6745,0.830360,0.915318,0.084957,0.232002,0.075139,0.075139
4,failed_training_count,6745,0.015315,0.038605,0.023291,0.171905,0.055675,0.055675
5,salary_growth_percent,6745,9.063276,7.955284,-1.107993,-0.143127,-0.046355,0.046355
6,bonus_target,6745,9.810838,9.238481,-0.572357,-0.127821,-0.041398,0.041398
7,review_count,6745,1.361494,1.266501,-0.094994,-0.085408,-0.027661,0.027661
8,months_since_promotion,6745,21.889517,23.138546,1.249029,0.083022,0.026888,0.026888
9,base_salary,6745,95329.215752,92818.057285,-2511.158467,-0.080998,-0.026233,0.026233


In [8]:
categorical_signals = read_output("categorical_signal_checks.csv")
(
    categorical_signals
    .loc[categorical_signals["included_in_rate_check"]]
    .sort_values("rate_ratio_vs_baseline", ascending=False)
    .head(15)
)

,feature,category,sample_size,positive_cases,attrition_rate,baseline_attrition_rate,rate_ratio_vs_baseline,included_in_rate_check
25,job_level,4,216,35,0.162037,0.119051,1.361071,True
13,employment_type,Hourly,1126,164,0.145648,0.119051,1.223410,True
15,job_family,Manufacturing,965,138,0.143005,0.119051,1.201208,True
26,job_level,1,1209,168,0.138958,0.119051,1.167211,True
33,region,Southwest,1202,165,0.137271,0.119051,1.153044,True
0,city,Phoenix,1202,165,0.137271,0.119051,1.153044,True
5,department_name,Information Technology,684,93,0.135965,0.119051,1.142071,True
16,job_family,Customer Support,490,66,0.134694,0.119051,1.131395,True
6,department_name,Customer Support,490,66,0.134694,0.119051,1.131395,True
34,region,Northeast,726,94,0.129477,0.119051,1.087571,True


In [9]:
diagnostic_model = read_output("diagnostic_model_summary.csv")
diagnostic_folds = read_output("diagnostic_model_folds.csv")
display(diagnostic_model)
display(diagnostic_folds)

,metric,value
0,Mean ROC-AUC,0.629818
1,ROC-AUC standard deviation,0.023868
2,Mean PR-AUC,0.192915
3,PR-AUC standard deviation,0.014785
4,No-skill PR-AUC,0.119051
5,PR-AUC lift over no-skill,1.620438


,fold,roc_auc,pr_auc
0,1,0.672787,0.205122
1,2,0.627786,0.179213
2,3,0.604910,0.173890
3,4,0.610646,0.193590
4,5,0.632960,0.212760


## 5. Interpretation

- The Version 2 twelve-month positive rate is inside the configured 7%–13% range.
- Voluntary and involuntary causes remain stochastic rather than deterministic.
- The strongest individual numeric correlation remains well below the 0.40 limit.
- A five-fold Logistic Regression diagnostic reaches the configured minimum ROC-AUC without producing unrealistically easy prediction.
- PR-AUC exceeds the no-skill class-prevalence baseline.
- All feature-source cutoffs are on or before the snapshot date.

The diagnostic model is only a generator-quality test. Formal temporal splitting, model comparison, calibration, and threshold selection occur in later checkpoints.